# Inspect train candidate-bank differences

Compare the strict-extractive 0.5B and 3B banks for the train split. This is analysis only; it does not modify the banks or use dev/test data.

In [ ]:
from pathlib import Path
import json, re, collections
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if PROJECT_ROOT.name != 'Task-Structured Counterfactual Preference Mining':
    PROJECT_ROOT = Path('/home/dinh-tuan/Documents/Project/BioASQ_MRES/Task-Structured Counterfactual Preference Mining')
RUN_ROOT = PROJECT_ROOT / 'Artifacts/cse_dpo/candidate_banks/strict_extractive_qwen25_05b_3b_all_splits_s20_v2_seed3407'
BANKS = {
    '05b': RUN_ROOT/'qwen25_05b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
    '3b': RUN_ROOT/'qwen25_3b/train/adapter-best-evidence-mrr/candidate_bank.jsonl',
}
for p in BANKS.values(): assert p.exists(), p
print(BANKS)

In [ ]:
def norm(x):
    x = re.sub(r'\[(?:BE|EE)\]', ' ', x or '', flags=re.I)
    return re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9]+', ' ', x.casefold())).strip()

rows = {}
for key, path in BANKS.items():
    by_q = collections.defaultdict(list)
    with path.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                r=json.loads(line); by_q[r['question_id']].append(r)
    rows[key]=by_q
print({k:(len(v), sum(map(len,v.values()))) for k,v in rows.items()})
assert set(rows['05b']) == set(rows['3b'])

In [ ]:
records=[]
for qid in rows['05b']:
    a,b=rows['05b'][qid],rows['3b'][qid]
    sa={norm(r.get('raw_output','')) for r in a if norm(r.get('raw_output',''))}
    sb={norm(r.get('raw_output','')) for r in b if norm(r.get('raw_output',''))}
    records.append({'question_id':qid,'unique_05b':len(sa),'unique_3b':len(sb),'shared_unique_outputs':len(sa&sb),'only_05b':len(sa-sb),'only_3b':len(sb-sa),'union_unique_outputs':len(sa|sb),'jaccard':len(sa&sb)/len(sa|sb) if sa|sb else 1.0,'exact_surface_overlap':sum(norm(x.get('raw_output',''))==norm(y.get('raw_output','')) for x,y in zip(a,b))})
df=pd.DataFrame(records)
display(df.describe().round(3))
print('Questions with any disagreement:', int((df.only_05b+df.only_3b>0).sum()), '/', len(df))
print('Mean per-question Jaccard:', round(df.jaccard.mean(),3))
print('Total unique answer surfaces:', {k:len({norm(r.get('raw_output','')) for q in rows[k].values() for r in q if norm(r.get('raw_output',''))}) for k in rows})

In [ ]:
# Inspect the largest disagreements. Each row keeps both raw outputs and sample IDs.
dis=[]
for qid in rows['05b']:
    a,b=rows['05b'][qid],rows['3b'][qid]
    aa=collections.Counter(norm(r.get('raw_output','')) for r in a); bb=collections.Counter(norm(r.get('raw_output','')) for r in b)
    dis.append({'question_id':qid,'question':a[0].get('question_text'),'only_05b':list((aa-bb).elements()),'only_3b':list((bb-aa).elements()),'shared':list((aa&bb).elements())})
dis_df=pd.DataFrame(dis)
dis_df['disagreement']=dis_df.only_05b.str.len()+dis_df.only_3b.str.len()
display(dis_df.sort_values('disagreement',ascending=False).head(20).to_string(index=False))

In [ ]:
# Optional gold-alias hit rate, using the prepared train targets.
prepared = PROJECT_ROOT/'Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/trainer_prepared/train_prepared.json'
gold=collections.defaultdict(set)
for r in json.loads(prepared.read_text()):
    qid=r['id'].split('__',1)[0]
    gold[qid].add(norm(r.get('output','')))
for key in rows:
    hits=[]
    for qid, rs in rows[key].items():
        targets=gold.get(qid,set()); hits.append(any(norm(r.get('raw_output','')) in targets for r in rs))
    print(key, 'questions with at least one exact prepared-target hit:', sum(hits), '/', len(hits))

In [ ]:
# Plot the main per-question comparison.
fig, ax = plt.subplots(1,3,figsize=(15,4))
df[['unique_05b','unique_3b']].plot(kind='box', ax=ax[0], title='Unique surfaces/question')
df[['shared_unique_outputs','only_05b','only_3b']].sum().plot(kind='bar', ax=ax[1], title='Aggregate surface overlap')
df.jaccard.plot(kind='hist', bins=20, ax=ax[2], title='Per-question Jaccard', xlabel='Jaccard')
plt.tight_layout(); plt.show()
OUT=RUN_ROOT/'train_bank_difference_summary.csv'; df.to_csv(OUT,index=False); print('Saved:',OUT)